# Machine Learning — Complete End-to-End Example
## Dataset: Salaries

This notebook walks through the **entire ML workflow** from raw data to trained model.
Every step is commented in detail so you can use this as a reference.

**Goal:** Predict a person's salary based on physical and demographic features.

**Type:** Regression (salary is a continuous number)

---
**Steps:**
1. Load & explore the data
2. Clean & preprocess
3. Visualize
4. Split into train/test
5. Train model
6. Evaluate
7. Improve

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

%matplotlib inline

## 2. Load the Data

We load the CSV file into a pandas DataFrame.
A DataFrame is essentially a table — rows are people, columns are their properties.

In [ ]:
# Load the dataset
df = pd.read_csv('salaries.csv')

# Always start by looking at the first few rows to understand the structure
df.head()

## 3. Explore the Data (EDA — Exploratory Data Analysis)

Before touching the data, we need to understand what we're working with.

In [ ]:
# Shape tells us how many rows (people) and columns (features) we have
print(f"Shape: {df.shape}")
print(f"→ {df.shape[0]} people, {df.shape[1]} features\n")

# dtypes tells us the data type of each column
# 'object' means text/string — these can't go directly into an ML model
print("Data types:")
print(df.dtypes)

In [ ]:
# describe() gives us statistics for all numeric columns:
# count, mean, std (spread), min, 25%/50%/75% quartiles, max
# Use this to spot outliers (is the max value realistic?)
df.describe()

In [ ]:
# Check for missing values — missing data will crash most ML algorithms
print("Missing values per column:")
print(df.isnull().sum())

print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Check for duplicate rows — they can bias the model
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Check categorical columns — what categories exist?
print("Experience values:", df['Experience'].unique())
print("Gender values:", df['Gender'].unique())
print("Daltonic values:", df['Daltonic'].unique())

## 4. Visualize

Visualization helps us understand relationships between features and spot potential issues.

In [ ]:
# Pair plot: shows the relationship between every pair of numeric features
# Diagonal: distribution of each feature
# Off-diagonal: scatter plot between two features
# → Look for: linear trends, clusters, outliers
sns.pairplot(df.select_dtypes(include='number'))
plt.suptitle('Pair Plot — All Numeric Features', y=1.02)
plt.show()

In [ ]:
# Correlation heatmap: shows correlation coefficients between all numeric features
# Value range: -1 (perfect negative) to +1 (perfect positive)
# 0 = no linear relationship
# Dark red = strong positive correlation
# Dark blue = strong negative correlation
plt.figure(figsize=(10, 8))
sns.heatmap(
    df.select_dtypes(include='number').corr(),
    annot=True,        # show numbers in cells
    fmt='.2f',         # round to 2 decimal places
    cmap='coolwarm'    # color scheme: blue=negative, red=positive
)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Look at salary distribution by experience level
# This helps confirm whether Experience is a useful predictor for Salary
plt.figure(figsize=(8, 5))
sns.boxplot(x='Experience', y='Salary', data=df)
plt.title('Salary Distribution by Experience Level')
plt.show()

## 5. Preprocess the Data

Machine learning models require **all-numeric input**.
We need to:
1. Handle missing values in 'Daltonic'
2. Convert text categories to numbers (one-hot encoding)

In [ ]:
# Step 1: Fill missing values in 'Daltonic'
# The NaN values represent people without color vision deficiency
# We fill them with 'None' so they get their own category during encoding
df['Daltonic'] = df['Daltonic'].fillna('None')

print("Daltonic after filling NaN:")
print(df['Daltonic'].value_counts())

In [ ]:
# Step 2: One-hot encode all categorical columns
# pd.get_dummies creates a new binary column for each unique value
# Example: 'Experience' with values 'Junior'/'Senior'
# → becomes 'Experience_Junior' (0 or 1) and 'Experience_Senior' (0 or 1)
# This is necessary because a model can't understand 'Junior' as text
df = pd.get_dummies(df, columns=['Experience', 'Gender', 'Daltonic'])

# Convert True/False booleans to 1/0 integers
# get_dummies creates boolean columns by default
df = df.apply(lambda x: x.astype(int) if x.dtype == bool else x)

print("Columns after encoding:")
print(df.columns.tolist())

In [ ]:
# Verify everything looks clean
df.describe()

## 6. Split into Features (X) and Target (y)

- **X** = all features (everything the model uses as input)
- **y** = target (what we want to predict — Salary)

In [ ]:
# X contains all columns EXCEPT the target
# We drop 'Salary' because that's what we're trying to predict
X = df.drop(columns=['Salary'])

# y contains only the target column
y = df['Salary']

print(f"X shape: {X.shape}  → {X.shape[0]} samples, {X.shape[1]} features")
print(f"y shape: {y.shape}  → {y.shape[0]} target values")

## 7. Train/Test Split

We split the data so we can evaluate the model on data it has **never seen during training**.
- Training set (80%): the model learns from this
- Test set (20%): we use this to measure real-world performance

In [ ]:
# test_size=0.2 → 20% goes to test, 80% to training
# random_state=42 → fixed random seed so we always get the same split
#   (important for reproducibility — without this, results change every run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set:  {X_train.shape[0]} samples")
print(f"Test set:      {X_test.shape[0]} samples")

## 8. Train the Model

We use **Linear Regression** as our first model.

Linear Regression fits a line (or hyperplane) through the data:
`Salary = β₀ + β₁·Height + β₂·Weight + β₃·Experience_Senior + ...`

The model finds the best coefficients (β values) that minimize prediction error.

In [ ]:
# Initialize the model
# At this point, no learning has happened yet — we just created an empty model
model = LinearRegression()

# Train the model on the training data
# .fit() is where the actual learning happens
# The model finds the best coefficients for each feature
model.fit(X_train, y_train)

print("Model trained!")
print(f"Intercept (β₀): {model.intercept_:.2f}")
print("\nCoefficients (one per feature):")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature}: {coef:.2f}")

## 9. Evaluate the Model

We evaluate on both training and test data:
- **R² on train**: how well the model fit the training data
- **R² on test**: how well it generalizes to unseen data

If train score >> test score → **overfitting** (model memorized training data)

In [ ]:
# Generate predictions on both sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# R² Score: ranges from -∞ to 1.0
# 1.0 = perfect, 0.0 = predicts the mean, negative = worse than the mean
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

# MAE: Mean Absolute Error — average error in the same unit as Salary ($)
mae = mean_absolute_error(y_test, y_test_pred)

print(f"R² Train:  {r2_train:.4f}")
print(f"R² Test:   {r2_test:.4f}")
print(f"MAE Test:  ${mae:,.2f}")

gap = r2_train - r2_test
if gap > 0.1:
    print(f"\n⚠️  Gap of {gap:.3f} between train/test — possible overfitting")
else:
    print(f"\n✓  Small gap ({gap:.3f}) — model generalizes well")

In [ ]:
# Visualize: Actual vs. Predicted values
# A perfect model would have all points on the diagonal line
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, alpha=0.6, color='steelblue', label='Predictions')

# Draw perfect prediction line
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect prediction')

plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title(f'Actual vs. Predicted Salary (R² = {r2_test:.3f})')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Improve — Try More Training Data (90/10 Split)

In [ ]:
# Try giving the model more training data (90% instead of 80%)
# More training data often = better model
X_train90, X_test90, y_train90, y_test90 = train_test_split(
    X, y, test_size=0.1, random_state=42
)

model90 = LinearRegression()
model90.fit(X_train90, y_train90)

y_pred90 = model90.predict(X_test90)
r2_90 = r2_score(y_test90, y_pred90)

print(f"R² with 80% training data: {r2_test:.4f}")
print(f"R² with 90% training data: {r2_90:.4f}")

if r2_90 > r2_test:
    print("→ More training data improved the model")
else:
    print("→ More training data did not improve the model")

## 11. Feature Importance

Which features actually matter for predicting salary?
We can look at the absolute value of the coefficients.
Larger absolute coefficient = bigger influence on the prediction.

In [ ]:
# Create a DataFrame of feature importances
importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_,
    'Abs_Coefficient': abs(model.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print("Feature importance (sorted by absolute coefficient):")
print(importance.to_string(index=False))

In [ ]:
# Visualize feature importance
plt.figure(figsize=(10, 6))
colors = ['green' if c > 0 else 'red' for c in importance['Coefficient']]
plt.barh(importance['Feature'], importance['Coefficient'], color=colors)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value')
plt.title('Feature Coefficients\n(green = positive effect, red = negative effect)')
plt.tight_layout()
plt.show()

## Summary

**What we did:**
1. Loaded and explored the salaries dataset (200 rows, 11 columns)
2. Found and handled missing values in 'Daltonic'
3. One-hot encoded categorical columns (Experience, Gender, Daltonic)
4. Split data 80/20 into train/test
5. Trained a Linear Regression model
6. Evaluated with R² and MAE
7. Tried improving by increasing training data to 90%
8. Analyzed which features had the most influence

**Key insight from this dataset:**
Physical features (Height, Weight, BMI, etc.) had near-zero correlation with Salary.
The most influential features were likely Experience and Gender — which makes sense in a real-world context.

**Next steps to try:**
- Remove the irrelevant physical features and retrain
- Try KNeighborsRegressor
- Try cross-validation for a more reliable score estimate